# Reproducibility Notebook: Fock-Resolved Black-Box SQR Inference

This notebook reproduces the main outputs of the study end-to-end. The workflow is:

1. Set the tunable parameters in the next cell.
2. Optionally rerun the study and validation scripts.
3. Load the saved JSON outputs.
4. Inspect the headline metrics, exact non-identifiability construction, and saved figures.

The purpose is to make the main claims easy to re-check and easy to modify.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

from IPython.display import Image, Markdown, display

SCRIPT_DIR = Path.cwd()
STUDY_DIR = SCRIPT_DIR.parent
DATA_DIR = STUDY_DIR / "data"
FIGURES_DIR = STUDY_DIR / "figures"
RUN_STUDY_SCRIPT = SCRIPT_DIR / "run_study.py"
VALIDATE_SCRIPT = SCRIPT_DIR / "validate_results.py"

print(f"Notebook directory: {SCRIPT_DIR}")
print(f"Study directory: {STUDY_DIR}")

In [ ]:
# Tunable parameters
# PROFILE controls the saved study size and should normally stay on "full".
# Set RUN_STUDY or RUN_VALIDATION to True if you want the notebook to regenerate outputs.
# SHOW_FIGURES controls which saved figures are displayed later.

PROFILE = "full"
RUN_STUDY = False
RUN_VALIDATION = False

SHOW_FIGURES = [
    "protocol_schematic",
    "single_qubit_fidelity_scaling",
    "protocol_comparison",
    "coherence_witness_residuals",
    "robustness_sensitivity",
    "pulse_case_recovery",
    "full_state_nonuniqueness",
    "waveform_examples",
]

In [ ]:
env = os.environ.copy()
env["STUDY_PROFILE"] = PROFILE

if RUN_STUDY:
    subprocess.run([sys.executable, str(RUN_STUDY_SCRIPT)], check=True, cwd=SCRIPT_DIR, env=env)

if RUN_VALIDATION:
    subprocess.run([sys.executable, str(VALIDATE_SCRIPT)], check=True, cwd=SCRIPT_DIR, env=env)

In [ ]:
results = json.loads((DATA_DIR / "study_results.json").read_text(encoding="utf-8"))
summary = json.loads((DATA_DIR / "study_summary.json").read_text(encoding="utf-8"))
validation = json.loads((DATA_DIR / "validation_summary.json").read_text(encoding="utf-8"))

display(Markdown("## Headline Metrics"))
display(Markdown(
    "\n".join([
        f"- Best single-qubit MLE mean fidelity: **{summary['best_single_qubit_mle_mean_fidelity']:.4f}**",
        f"- Wait-only transverse rank: **{summary['identifiability']['wait_only']['transverse_rank']}**",
        f"- Displacement-only transverse rank: **{summary['identifiability']['displacement_only']['transverse_rank']}**",
        f"- Coherent-case residual: wait-only **{summary['coherence_wait_residual']:.2e}**, combined **{summary['coherence_combined_residual']:.3f}**",
        f"- Exact gauge-family constructions: **{summary['exact_gauge_family_count']}**",
        f"- Validation checks all passed: **{validation['all_passed']}**",
    ])
))

In [ ]:
display(Markdown("## Exact Non-Identifiability Construction"))
for index, solution in enumerate(results["exact_gauge_family"], start=1):
    display(Markdown(f"### Gauge-family solution {index}"))
    print("probabilities:", [round(value, 6) for value in solution["probabilities"]])
    for level, bloch in enumerate(solution["bloch_rows"]):
        print(f"  n={level}: x={bloch['x']:.6f}, y={bloch['y']:.6f}, z={bloch['z']:.6f}")
    print("reconstructed z_total:", round(solution["reconstructed_z_total"], 6))
    print()

In [ ]:
display(Markdown("## Saved Figures"))
for stem in SHOW_FIGURES:
    png_path = FIGURES_DIR / f"{stem}.png"
    if png_path.exists():
        display(Markdown(f"### {stem.replace('_', ' ').title()}"))
        display(Image(filename=str(png_path)))
    else:
        print(f"Missing figure: {png_path}")